# Basic LangGraph example with a simple agent that can use tools

In [1]:
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import HumanMessage, BaseMessage
from langchain.tools import tool
from langchain_google_vertexai import ChatVertexAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import ToolMessage

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

tools = [multiply, add]

llm = ChatVertexAI(model_name="gemini-2.5-flash", temperature=0.0)
llm_with_tools = llm.bind_tools(tools)

In [2]:
def agent_node(state: AgentState) -> AgentState:
    """
    Run the agent LLM with tools on the current state messages and append the AI message to the state.

    Args:
        state (AgentState): The current agent state containing the message history.

    Returns:
        AgentState: Updated state with the new AI message appended.
    """
    ai_msg = llm_with_tools.invoke(state["messages"])
    return {"messages": [*state["messages"], ai_msg]}

def route_to_tools(state: AgentState) -> str:
    """
    Route to the 'tools' node if the last message contains tool calls, otherwise end the graph.

    Args:
        state (AgentState): The current agent state containing the message history.

    Returns:
        str: 'tools' if tool calls are present, otherwise END.
    """
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return END

def tools_node(state: AgentState) -> AgentState:
    """
    Execute all tool calls in the last AI message and append their results as ToolMessages to the state.

    Args:
        state (AgentState): The current agent state containing the message history.

    Returns:
        AgentState: Updated state with tool outputs appended as ToolMessages.
    """
    last = state["messages"][-1]
    new_msgs = []
    # iterate over all tool calls in the last AI message
    for call in getattr(last, "tool_calls", []):
        # Find the matching tool by its name (StructuredTool exposes .name)
        tool_name = call["name"]
        tool_obj = next(t for t in tools if t.name == tool_name)

        # Execute the tool with the arguments supplied by the LLM
        result = tool_obj.invoke(call["args"])

        # Append a ToolMessage, including the original tool_call_id
        new_msgs.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"],
            )
        )

    # Return updated state with the AI message and tool output(s)
    return {"messages": [*state["messages"], *new_msgs]}

Create graph:

In [3]:
graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tools_node)
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", route_to_tools)
graph.add_edge("tools", "agent")

app = graph.compile()

In [4]:
query = "Multiply 20 by 5 and add 10 to the result."

final_state = app.invoke({"messages": [HumanMessage(content=query)]})
print(final_state["messages"][-1].content)

The result is 110.


### Use MCP to run the agent

In [3]:
import nest_asyncio
nest_asyncio.apply()  # allow nested event loops in Jupyter

from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_google_vertexai import ChatVertexAI
from langgraph.prebuilt import create_react_agent

async def run_agent(query: str) -> str:
    # configuration for your running MCP server
    mcp_config = {
        "mytools": {
            "command": "python",
            "args": ["/Users/avosseler/Github/Team/ai-assistant-backend-common-core/src/agbe_cc/resources/toy_mcp.py"],  # path to your MCP server script
            "transport": "stdio",
        }
    }

    client = MultiServerMCPClient(mcp_config)
    
    tools = await client.get_tools(server_name="mytools")

    llm = ChatVertexAI(model_name="gemini-2.5-flash", temperature=0.0)
    
    agent = create_react_agent(llm, tools)

    # ask a question; the agent will call your MCP tool under the hood
    response = await agent.ainvoke({
        "messages": [HumanMessage(content=query)]
    })

    # extract the final answer from the messages list
    return response["messages"][-1].content

In [4]:
query = "Multiply 20 by 5 and add 10 to the result."

result = await run_agent(query=query)

I0000 00:00:1755262262.857551  941257 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0000 00:00:1755262264.492156  941257 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


In [5]:
print(result)

The result is 110.


# Other version using LangGraph to run the agent with tools (recommended):

In [ ]:
import nest_asyncio

nest_asyncio.apply()

from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import HumanMessage, AIMessage
from langchain_google_vertexai import ChatVertexAI
from langchain.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph.message import add_messages
from mcp_sandbox.config.config import model_list

class AgentState(TypedDict):
    messages: Annotated[Sequence[AIMessage | HumanMessage], add_messages]

# Define local tools (i.e. Non-MCP server tools)
@tool
def local_divide(a: float, b: float) -> float:
    """Divide two numbers."""
    if b == 0:
        return float('inf')
    return a / b

@tool
def local_power(a: float, b: float) -> float:
    """Raise a to the power of b."""
    return a ** b

# Dockerized MCP tools setup
async def setup_tools():
    # Get MCP tools from your containerized server
    mcp_client = MultiServerMCPClient({
        "mytools": {
            "command": "docker",
            "args": ["exec", "-i", "mcp-server-1", "uv", "run", "python", "src/mcp_sandbox/resources/mcp_server.py"],
            "transport": "stdio",
        }
    })
    mcp_tools = await mcp_client.get_tools(server_name="mytools")
    
    # Combine local tools with MCP tools
    local_tools = [local_divide, local_power]
    all_tools = local_tools + mcp_tools
    
    return all_tools

# -------------------------------------------------------------------
# # Setup MCP tools from a local script
# async def setup_tools():
#     client = MultiServerMCPClient({
#         "mytools": {
#             "command": "python",
#             "args": ["/Users/avosseler/Github/Privat/mcp-playground/src/mcp_sandbox/resources/mcp_server.py"],  
#             "transport": "stdio",
#         }
#     })
#     mcp_tools = await client.get_tools(server_name="mytools")
    
#     # Combine local tools with MCP tools
#     local_tools = [local_divide, local_power]
#     all_tools = local_tools + mcp_tools
    
#     return all_tools

# Use the combined tools
all_tools = await setup_tools()

llm = ChatVertexAI(model_name=model_list["chat_model"]["google"], temperature=0.1)

model_with_tools = llm.bind_tools(all_tools)

def call_model(state: AgentState) -> AgentState:
    print("🤖 Calling model...")
    response = model_with_tools.invoke(state["messages"])
    print(f"🤖 Model response: {response.content[:100]}...")
    if hasattr(response, 'tool_calls') and response.tool_calls:
        print(f"🔧 Tool calls requested: {[call['name'] for call in response.tool_calls]}")
    return {"messages": [response]}

# Executes tools requested by the LLM
tool_node = ToolNode(all_tools)

graph = StateGraph(AgentState)
graph.add_node("call_model", call_model)
graph.add_node("tools", tool_node)
graph.add_edge(START, "call_model")
# If the last AIMessage contains any tool_calls, tools_condition routes execution to the tools node; otherwise, execution ends.
# If tools are requested, go to tools
graph.add_conditional_edges("call_model", tools_condition)
# After tool execution, return to call_model for final integration
graph.add_edge("tools", "call_model")
graph.set_finish_point("call_model")

app = graph.compile()

In [3]:
# query = "Multiply 20 by 5 and add 10 and finally square the result."
query = "Calculate: 2+3-1 and finally square the result."

Run agent:

In [4]:
print(f"💬 Query: {query}")

final = await app.ainvoke({
    "messages": [HumanMessage(content=query)]
})

print(final["messages"][-1].content)

💬 Query: Calculate: 2+3-1 and finally square the result.
🤖 Calling model...
🤖 Model response: ...
🔧 Tool calls requested: ['add']
🤖 Calling model...
🤖 Model response: ...
🔧 Tool calls requested: ['add']
🤖 Calling model...
🤖 Model response: ...
🔧 Tool calls requested: ['local_power']
🤖 Calling model...
🤖 Model response: The result is 16....
The result is 16.


In [4]:
# query = "Multiply 20 by 5 and add 10 to the result."

# # Stream the execution to see each step
# for z, step in enumerate(app.stream({"messages": [HumanMessage(content=query)]})):
#     print(f"Step {z}: {step}")
#     print("---")

In [11]:
print(f"💬 Query: {query}")
async for step in app.astream({"messages": [HumanMessage(content=query)]}):
    print(f"Step: {step}")
    print("---")

💬 Query: Calculate: 2+3-1 and finally square the result.
🤖 Calling model...
🤖 Model response: ...
🔧 Tool calls requested: ['add']
Step: {'call_model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"b": 3.0, "a": 2.0}'}, '__gemini_function_call_thought_signatures__': {'d15ff233-f00c-42ed-99cc-9fcfd499be5c': 'CvQFAcu98PCSakRYIrsxg88g9BGGzVLtMqH7ga+qNfehXLefFQGYtmV+rn+SDQ/5FPSo1wM1ypfujEJtonU2ux7rZARkG+gztqWZhiCTb2sfPbeObHVGVOToBFXTGnxjDwDAyHoL62PHWYMNegD+Jbw7mSHWLpz+m3Rq+UDJIAJpTVmyx9Cy1sFx+acut84vTzc7IrlnO5s6xzIFbjuA1wM9lmJaGf2mw4lj8aD8w6a78bcwlAKdB0NRCAer0lRwcxvf8jz0y2LeUsm3bEgTIqwvv+Act984YCFc0WEoXNQgBQF0HlbFQqwrlMmA8QwrpFWQZl4zVucKlruJgiFGh1Hk8iWchodT+YP8KMGVz7UZRGVPHnAfXeG7pfn3ZbwX3MJ5BCtIUasWGezVpZEEUxbkFlvnte2rOcekP21qK0zGUfDO2rD5jDtXpAitXyBmRxI8IetZ9Nj4akWfRP80U8idO4elchdKsUp+9beWNzd1H2JWhm50JtDF1mnU+aNczWjxHfqS9FzInhgQctNRUahMUkYlJwh4Bc7/Tfc/SW2+Uhg/IGADPlpDq9NKginnqea/IoNagL7bqlmJfXwn9b6AhmGj7hAmSraV+p/FsXTpKmrc0t0WhbHKN8

In [ ]:
import requests

query = "Multiply 2 by 5 and add 2 and finally square the result."
#query = "Calculate: 2+3-1 and finally square the result."

response = requests.post(
    'http://localhost:8080/calculate',
    headers={'Content-Type': 'application/json'},
    json={'query': query}, verify=False, timeout=60
)

result = response.json()
print('Answer:', result.get('answer', 'No answer found'))

Answer: The result is 144.0.


In [1]:
# Call a cloud function for 'addition':
# import subprocess
# import requests
# import json

# def get_id_token():
#     """Get ID token using gcloud in a subprocess"""
#     result = subprocess.run(
#         ["gcloud", "auth", "print-identity-token"],
#         stdout=subprocess.PIPE,
#         text=True,
#         check=True
#     )
#     return result.stdout.strip()

# id_token_val = get_id_token()

# url = "https://addition-376814595095.europe-west3.run.app"

# headers = {
#     "Authorization": f"bearer {id_token_val}",
#     "Content-Type": "application/json"
# }

# payload = {"a": 10, "b": -3.5}

# response = requests.post(url, json=payload, headers=headers, verify=False)

# out = json.loads(response.text)
# print(f"Sum of {payload['a']} and {payload['b']} is: {out["sum"]}")